In [13]:
!pip install pandas mysql-connector-python

import pandas as pd
import mysql.connector
from mysql.connector import Error
import unicodedata

#from google.colab import files
#from google.colab import drive

In [16]:
# Carregar o arquivo Excel/CSV em um dataframe

# Se arquivo for local:
file_path_leitura = 'C:\\Users\\NARCI\\OneDrive\\Desktop\\Univesp - PI\\CAPES - IES - Brasil (GEOREFERENCIADA) V6.csv'
file_path_escrita = 'C:\\Users\\NARCI\\OneDrive\\Desktop\\Univesp - PI\\CAPES - IES - Brasil (GEOREFERENCIADA) limpo V6.csv'
#Se o arquivo estiver no Drive:
# drive.mount('/content/drive')

#df = pd.read_csv(file_path_leitura, encoding='utf-8-sig')
df = pd.read_csv(file_path_leitura)
print("DataFrame carregado com sucesso!")

# Limpeza básica: remove espaços em branco no início/fim das strings para evitar duplicatas falsas
#df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
#print("Limpeza realizada com sucesso!")

# Remover acentos
# Função para remover acentos
def remover_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

# Aplicar a função Remover Acentos na coluna desejada
df['IES - SIGLA'] = df['IES - SIGLA'].apply(remover_acentos)
df['IES - NOME'] = df['IES - NOME'].apply(remover_acentos)
df['Endereço'] = df['Endereço'].apply(remover_acentos)
df['Município'] = df['Município'].apply(remover_acentos)
df['Status Jurídico'] = df['Status Jurídico'].apply(remover_acentos)
df['Áreas de pesquisa'] = df['Áreas de pesquisa'].apply(remover_acentos)
df['Temas específicos de pequisa'] = df['Temas específicos de pequisa'].apply(remover_acentos)
df['Área Conhecimento'] = df['Área Conhecimento'].apply(remover_acentos)
df['Grande Área'] = df['Grande Área'].apply(remover_acentos)
df['Especialização'] = df['Especialização'].apply(remover_acentos)
df['Mestrado'] = df['Mestrado'].apply(remover_acentos)
df['Doutorado'] = df['Doutorado'].apply(remover_acentos)
print("Remoção de acentos concluída com sucesso!")

# Substituir ç por c e ç por C
df = df.replace(to_replace='ç', value='c', regex=True)
df = df.replace(to_replace='Ç', value='C', regex=True)
print("Remoção de Ç concluída com sucesso!")

# Substitui vírgulas por pontos e converte para float, se necessário
df['Latitude'] = df['Latitude'].astype(str).str.replace(',', '.').astype(float)
df['Longitude'] = df['Longitude'].astype(str).str.replace(',', '.').astype(float)
print("Substituição de vírgulas por pontos concluída com sucesso!")

total_linhas = len(df)
print(f"Total de linhas: {total_linhas}")

# Salva uma cópia do dataframe em arquivo CSV sem acentos e ç
#df.to_csv(file_path_escrita, index=False, encoding='utf-8-sig')
df.to_csv(file_path_escrita, index=False)
print("Criado CSV limpo com sucesso!")
print(df.head(2))

DataFrame carregado com sucesso!
Remoção de acentos concluída com sucesso!
Remoção de Ç concluída com sucesso!
Substituição de vírgulas por pontos concluída com sucesso!
Total de linhas: 8558
Criado CSV limpo com sucesso!
  IES - SIGLA                        IES - NOME  \
0        UNIR  UNIVERSIDADE FEDERAL DE RONDONIA   
1        UNIR  UNIVERSIDADE FEDERAL DE RONDONIA   

                                            Endereço    Município  UF  \
0  Av. Pres. Dutra, 2965 - Olaria, Porto Velho - ...  PORTO VELHO  RO   
1  Av. Pres. Dutra, 2965 - Olaria, Porto Velho - ...  PORTO VELHO  RO   

   Latitude  Longitude                          Homepage Status Jurídico  \
0 -8.763544 -63.906526  https://propesq.unir.br/homepage         Federal   
1 -8.763544 -63.906526  https://propesq.unir.br/homepage         Federal   

                                 Áreas de pesquisa  \
0  BIOLOGIA EXPERIMENTAL - CIENCIAS BIOLOGICAS III   
1  BIOLOGIA EXPERIMENTAL - CIENCIAS BIOLOGICAS III   

  Temas espe

In [17]:
# Configurações de Conexão com o Banco de Dados
db_config = {
    'host': 'localhost',      # 'localhost' ou o IP do seu servidor MySQL
    'user': 'root',           # seu usuário
    'password': '1234',  # sua senha
    'database': 'mydb', # Nome do banco de dados
}

try:
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
   
    # Dicionários de "Cache" (Evita consultas desnecessárias ao banco e acelera o script)
    map_estados = {}
    map_cidades = {}
    map_ies = {}
    map_enderecos = {}
    map_status = {}
    map_modalidades = {}
    map_ga = {}
    map_ac = {}
    map_ap = {}
    map_temas = {}
    estados = {"AC": "Acre", "AL": "Alagoas", "AP": "Amapa", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceara", "DF": "Distrito Federal", "ES": "Espirito Santo", "GO": "Goias", "MA": "Maranhao", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul", "MG": "Minas Gerais", "PA": "Para", "PB": "Paraíba", "PR": "Parana", "PE": "Pernambuco", "PI": "Piaui", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondonia", "RR": "Roraima", "SC": "Santa Catarina", "SP": "Sao Paulo", "SE": "Sergipe", "TO": "Tocantins"}

    print("Iniciando a inserção dos dados (isso pode levar alguns segundos)...")

    # Itera sobre cada linha do CSV
    for index, row in df.iterrows():
        print(f"Processando linha {index + 1}/{total_linhas} - IES: {row['IES - NOME']}") # Log para acompanhar o progresso
        # ==========================================
        # 1. ESTADOS
        # ==========================================
        uf = row['UF'] if pd.notna(row['UF']) else 'ND'
        if uf not in map_estados:
            cursor.execute("SELECT estado_id FROM estados WHERE sigla_estado = %s", (uf,))
            res = cursor.fetchone()
            if res:
                map_estados[uf] = res[0]
            else:
                cursor.execute("INSERT INTO estados (sigla_estado, estado) VALUES (%s, %s)", (uf, estados.get(uf, 'Não Informado')))
                map_estados[uf] = cursor.lastrowid
        estado_id = map_estados[uf]

        # ==========================================
        # 2. CIDADES
        # ==========================================
        municipio = row['Município'] if pd.notna(row['Município']) else 'Não Informado'
        cidade_key = (estado_id, municipio)
        if cidade_key not in map_cidades:
            cursor.execute("SELECT cidade_id FROM cidades WHERE cidade = %s AND estados_estado_id = %s", (municipio, estado_id))
            res = cursor.fetchone()
            if res:
                map_cidades[cidade_key] = res[0]
            else:
                cursor.execute("INSERT INTO cidades (cidade, estados_estado_id) VALUES (%s, %s)", (municipio, estado_id))
                map_cidades[cidade_key] = cursor.lastrowid
        cidade_id = map_cidades[cidade_key]

        # ==========================================
        # 3. INSTITUIÇÕES DE ENSINO SUPERIOR (IES)
        # ==========================================
        ies_nome = row['IES - NOME']
      
        if ies_nome not in map_ies:
            cursor.execute("SELECT ies_id FROM instituicoes_ensino_superior WHERE nome_ies = %s", (ies_nome,))
            res = cursor.fetchone()
            if res:
                map_ies[ies_nome] = res[0]
            else:
                sigla = row['IES - SIGLA'] if pd.notna(row['IES - SIGLA']) else None
                homepage = row['Homepage'] if pd.notna(row['Homepage']) else None
                cursor.execute("INSERT INTO instituicoes_ensino_superior (sigla_ies, nome_ies, homepage) VALUES (%s, %s, %s)", (sigla, ies_nome, homepage))
                map_ies[ies_nome] = cursor.lastrowid
        ies_id = map_ies[ies_nome]

        # ==========================================
        # 4. ENDEREÇOS IES
        # ==========================================
        endereco = row['Endereço'] if pd.notna(row['Endereço']) else 'Não Informado'
        end_key = (ies_id, endereco)
        if end_key not in map_enderecos:
            cursor.execute("SELECT endereco_id FROM enderecos_ies WHERE instituicoes_ensino_superior_ies_id = %s AND logradouro = %s", (ies_id, endereco))
            res = cursor.fetchone()
            if res:
                map_enderecos[end_key] = res[0]
            else:
                lat = row['Latitude'] if pd.notna(row['Latitude']) else None
                lon = row['Longitude'] if pd.notna(row['Longitude']) else None
                cursor.execute("INSERT INTO enderecos_ies (logradouro, latitude, longitude, cidades_cidade_id, instituicoes_ensino_superior_ies_id) VALUES (%s, %s, %s, %s, %s)", (endereco, lat, lon, cidade_id, ies_id))
                map_enderecos[end_key] = cursor.lastrowid

        # ==========================================
        # 5. STATUS JURÍDICO
        # ==========================================
        status = row['Status Jurídico'] if pd.notna(row['Status Jurídico']) else None
        if status:
            status_key = (ies_id, status)
            if status_key not in map_status:
                cursor.execute("SELECT status_juridico_id FROM status_juridico WHERE status_juridico = %s AND instituicoes_ensino_superior_ies_id = %s", (status, ies_id))
                if not cursor.fetchone():
                    cursor.execute("INSERT INTO status_juridico (status_juridico, instituicoes_ensino_superior_ies_id) VALUES (%s, %s)", (status, ies_id))
                map_status[status_key] = True

        # ==========================================
        # 6. MODALIDADES DE PÓS GRADUAÇÃO
        # ==========================================
        for col, nome_mod in [('Especialização', 'Especialização'), ('Mestrado', 'Mestrado'), ('Doutorado', 'Doutorado')]:
            if pd.notna(row[col]) and row[col].strip().upper() == 'SIM':
                mod_key = (ies_id, nome_mod)
                if mod_key not in map_modalidades:
                    cursor.execute("SELECT modalide_id FROM modalidades_pos_graduacao WHERE modalidade_pos_graduacao = %s AND instituicoes_ensino_superior_ies_id = %s", (nome_mod, ies_id))
                    if not cursor.fetchone():
                        cursor.execute("INSERT INTO modalidades_pos_graduacao (modalidade_pos_graduacao, instituicoes_ensino_superior_ies_id) VALUES (%s, %s)", (nome_mod, ies_id))
                    map_modalidades[mod_key] = True

        # ==========================================
        # 7. GRANDES ÁREAS
        # ==========================================
        grande_area = row['Grande Área'] if pd.notna(row['Grande Área']) else 'Não informada'
        ga_key = (ies_id, grande_area)
        if ga_key not in map_ga:
            cursor.execute("SELECT grande_area_id FROM grandes_areas WHERE grande_area = %s AND instituicoes_ensino_superior_ies_id = %s", (grande_area, ies_id))
            res = cursor.fetchone()
            if res:
                map_ga[ga_key] = res[0]
            else:
                cursor.execute("INSERT INTO grandes_areas (grande_area, instituicoes_ensino_superior_ies_id) VALUES (%s, %s)", (grande_area, ies_id))
                map_ga[ga_key] = cursor.lastrowid
        ga_id = map_ga[ga_key]

        # ==========================================
        # 8. ÁREAS DE CONHECIMENTO
        # ==========================================
        area_conh = row['Área Conhecimento'] if pd.notna(row['Área Conhecimento']) else 'Não informada'
        ac_key = (ga_id, area_conh)
        if ac_key not in map_ac:
            cursor.execute("SELECT area_conhecimento_id FROM areas_conhecimento WHERE area_conhecimento = %s AND grandes_areas_grande_area_id = %s", (area_conh, ga_id))
            res = cursor.fetchone()
            if res:
                map_ac[ac_key] = res[0]
            else:
                cursor.execute("INSERT INTO areas_conhecimento (area_conhecimento, grandes_areas_grande_area_id) VALUES (%s, %s)", (area_conh, ga_id))
                map_ac[ac_key] = cursor.lastrowid
        ac_id = map_ac[ac_key]

        # ==========================================
        # 9. ÁREAS DE PESQUISA
        # ==========================================
        area_pesq = row['Áreas de pesquisa'] if pd.notna(row['Áreas de pesquisa']) else 'Não informada'
        ap_key = (ac_id, area_pesq)
        if ap_key not in map_ap:
            cursor.execute("SELECT area_pesquisa_id FROM areas_pesquisa WHERE area_pesquisa = %s AND areas_conhecimento_area_conhecimento_id = %s", (area_pesq, ac_id))
            res = cursor.fetchone()
            if res:
                map_ap[ap_key] = res[0]
            else:
                cursor.execute("INSERT INTO areas_pesquisa (area_pesquisa, areas_conhecimento_area_conhecimento_id) VALUES (%s, %s)", (area_pesq, ac_id))
                map_ap[ap_key] = cursor.lastrowid
        ap_id = map_ap[ap_key]

        # ==========================================
        # 10. TEMAS ESPECÍFICOS DE PESQUISA
        # ==========================================
        tema = row['Temas específicos de pequisa'] if pd.notna(row['Temas específicos de pequisa']) else None
        if tema:
            tema_key = (ap_id, tema)
            if tema_key not in map_temas:
                cursor.execute("SELECT temas_pesquisa_id FROM temas_especificos_pesquisa WHERE tema_pesquisa = %s AND areas_pesquisa_area_pesquisa_id = %s", (tema, ap_id))
                if not cursor.fetchone():
                    cursor.execute("INSERT INTO temas_especificos_pesquisa (tema_pesquisa, areas_pesquisa_area_pesquisa_id) VALUES (%s, %s)", (tema, ap_id))
                map_temas[tema_key] = True

    # Confirma todas as inserções no banco de dados
    conn.commit()
    print("Sucesso! Banco de dados atualizado mantendo todas as amarrações do seu modelo (MWB).")

except Error as e:
    print(f"Erro no banco de dados MySQL: {e}")
    if 'conn' in locals() and conn.is_connected():
        conn.rollback() # Desfaz as operações em caso de erro para não quebrar a integridade referencial
        print("Operação desfeita (Rollback).")
        print(df.iloc[index]) # Imprime a linha problemática para ajudar na depuração
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()

Iniciando a inserção dos dados (isso pode levar alguns segundos)...
Processando linha 1/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 2/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 3/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 4/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 5/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 6/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 7/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 8/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 9/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 10/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 11/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 12/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 13/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha 14/8558 - IES: UNIVERSIDADE FEDERAL DE RONDONIA
Processando linha

Agora, vamos converter o DataFrame em uma matriz NumPy.